# 

In [2]:
#!/usr/bin/env python
# coding: utf-8

import os
import glob
import pickle
import numpy as np
import pandas as pd
import mne


from EEGPrep import finalPreprocesseeg
from EYEPrep import finalPreprocesseye


BASE_PATH = r"E:\koorathota_data\pkl_data\pkl_data\pkl_data"
OUTPUT_DIR = r"E:\koorathota_data\pkl_data\pkl_data\pkl_data\eegdata"

os.makedirs(OUTPUT_DIR, exist_ok=True)

pkl_files = glob.glob(os.path.join(BASE_PATH, "*.pkl"))

all_X_eeg = []
all_y_cls = []
all_y_reg = []
all_subject_id_eeg = []
all_session_id_eeg = []

all_stats = []

all_X_eye_L = []
all_X_eye_R = []
all_X_eye_M = []

all_y_cls_eye_L = []
all_y_cls_eye_R = []
all_y_cls_eye_M = []

all_y_reg_eye_L = []
all_y_reg_eye_R = []
all_y_reg_eye_M = []

all_subject_id_L= []
all_subject_id_R =[]
all_subject_id_M =[]

all_session_id_L=[]
all_session_id_R=[]
all_session_id_M=[]
all_eeg_times = []
all_times_L =[]
all_times_R =[]
all_times_M=[]
reference_ch_names = None
reference_shape = None


print(f"Found {len(pkl_files)} files to process.")



for file_path in pkl_files:
    file_name = os.path.basename(file_path)
    print(f"\nProcessing: {file_name}")

    try:
        with open(file_path, "rb") as f:
            data = pickle.load(f)

        resulteeg = finalPreprocesseeg(data)

        if resulteeg is None:
            raise ValueError(f"Missing required keys in {file_name}")

        (
            raw_ica,
            epochs_ar,
            reject_log,
            filtered_left_peaks,
            filtered_right_peaks,
            filtered_left_onsets,
            filtered_right_onsets,
            events,
            y_class,
            y_reg,
            stats,
        ) = resulteeg

        resulteye = finalPreprocesseye(data, events)

        if resulteye is None:
            raise ValueError(f"Missing required keys in {file_name}")

        (
            pupil_epochsL,
            epoch_infoL,
            rejectedL,
            rel_timesL,
            kept_indicesL,
            pupil_epochsR,
            epoch_infoR,
            rejectedR,
            rel_timesR,
            kept_indicesR,
            pupil_epochsM,
            epoch_infoM,
            rejectedM,
            rel_timesM,
            kept_indicesM,
        ) = resulteye

        # Fill residual NaNs in pupil epochs
        nan_mask = ~np.isfinite(pupil_epochsL)
        if nan_mask.any():
            print(f"  Filling {nan_mask.sum()} residual NaN samples in left pupil with 0")
        pupil_epochsL[nan_mask] = 0.0

        nan_mask = ~np.isfinite(pupil_epochsR)
        if nan_mask.any():
            print(f"  Filling {nan_mask.sum()} residual NaN samples in right pupil with 0")
        pupil_epochsR[nan_mask] = 0.0

        nan_mask = ~np.isfinite(pupil_epochsM)
        if nan_mask.any():
            print(f"  Filling {nan_mask.sum()} residual NaN samples in mean pupil with 0")
        pupil_epochsM[nan_mask] = 0.0

        # Eye tensors: [N, T, 1]
        X_eye_L = pupil_epochsL[:, :, np.newaxis].astype(np.float32)
        X_eye_R = pupil_epochsR[:, :, np.newaxis].astype(np.float32)
        X_eye_M = pupil_epochsM[:, :, np.newaxis].astype(np.float32)
        n_epochs_L = X_eye_L.shape[0]
        n_epochs_R = X_eye_R.shape[0]
        n_epochs_M = X_eye_M.shape[0]
        # EEG labels
        keep_mask = ~reject_log.bad_epochs
        events_eeg = events[keep_mask]
        event_sample = events_eeg[:, 0].astype(np.int64)
        event_times = event_sample/(epochs_ar.info['sfreq'])
        y_cls_eeg = np.asarray(y_class[keep_mask], dtype=np.int64)
        y_reg_eeg = np.asarray(y_reg[keep_mask], dtype=np.float32)

        # Eye labels aligned to kept indices
        y_cls_eye_L = y_class[kept_indicesL]
        y_cls_eye_R = y_class[kept_indicesR]
        y_cls_eye_M = y_class[kept_indicesM]

        y_reg_eye_L = y_reg[kept_indicesL]
        y_reg_eye_R = y_reg[kept_indicesR]
        y_reg_eye_M = y_reg[kept_indicesM]

        events_L = events[kept_indicesL]
        events_R = events[kept_indicesR]
        events_M = events[kept_indicesM]

        times_L = np.array([d["event_time"] for d in epoch_infoL], dtype=np.float64)
        times_R = np.array([d["event_time"] for d in epoch_infoR], dtype=np.float64)
        times_M = np.array([d["event_time"] for d in epoch_infoM], dtype=np.float64)

        
        # EEG tensor
        epochs_reve = epochs_ar.copy().pick("eeg")
        epochs_reve = epochs_reve.copy().resample(200)

        X_eeg = epochs_reve.get_data(copy=True).astype(np.float32)  # [N, C, T]
        ch_names = epochs_reve.ch_names
        n_epochs = X_eeg.shape[0]

        if len(y_cls_eeg) != n_epochs or len(y_reg_eeg) != n_epochs:
            raise ValueError(
                f"Label length mismatch in {file_name}: "
                f"n_epochs={n_epochs}, len(y_cls_eeg)={len(y_cls_eeg)}, len(y_reg_eeg)={len(y_reg_eeg)}"
            )

        if reference_ch_names is None:
            reference_ch_names = ch_names
        elif ch_names != reference_ch_names:
            raise ValueError(f"Channel mismatch in {file_name}")

        if reference_shape is None:
            reference_shape = X_eeg.shape[1:]  # (C, T)
        elif X_eeg.shape[1:] != reference_shape:
            raise ValueError(
                f"Shape mismatch in {file_name}: {X_eeg.shape[1:]} vs {reference_shape}"
            )

        # Subject / session metadata
        info = data["Unity_TrialInfo"]
        info_t = info[0]
        idd = info_t[0]
        session = info_t[1]

        subj_value = idd[0]
        sess_value = session[0]
    
        subject_id_eeg = np.full(n_epochs, subj_value, dtype=np.int64)
        session_id_eeg = np.full(n_epochs, sess_value, dtype=np.int64)
        subject_id_eye_L = np.full(n_epochs_L, subj_value, dtype=np.int64)
        session_id_eye_L = np.full(n_epochs_L, sess_value, dtype=np.int64)
        subject_id_eye_R = np.full(n_epochs_R, subj_value, dtype=np.int64)
        session_id_eye_R = np.full(n_epochs_R, sess_value, dtype=np.int64)
        subject_id_eye_M = np.full(n_epochs_M, subj_value, dtype=np.int64)
        session_id_eye_M = np.full(n_epochs_M, sess_value, dtype=np.int64)
        
        # Store EEG
        all_X_eeg.append(X_eeg)
        all_y_cls.append(y_cls_eeg)
        all_y_reg.append(y_reg_eeg)
        all_subject_id_eeg.append(subject_id_eeg)
        all_session_id_eeg.append(session_id_eeg)
        all_eeg_times.append(event_times)
        # Store Eye
        all_X_eye_L.append(X_eye_L)
        all_X_eye_R.append(X_eye_R)
        all_X_eye_M.append(X_eye_M)

        all_y_cls_eye_L.append(y_cls_eye_L)
        all_y_cls_eye_R.append(y_cls_eye_R)
        all_y_cls_eye_M.append(y_cls_eye_M)

        all_y_reg_eye_L.append(y_reg_eye_L)
        all_y_reg_eye_R.append(y_reg_eye_R)
        all_y_reg_eye_M.append(y_reg_eye_M)
        all_subject_id_L.append(subject_id_eye_L)
        all_session_id_L.append(session_id_eye_L)
        all_subject_id_R.append(subject_id_eye_R)
        all_session_id_R.append(session_id_eye_R)
        all_subject_id_M.append(subject_id_eye_M)
        all_session_id_M.append(session_id_eye_M)
        all_times_L.append(times_L)
        all_times_R.append(times_R)
        all_times_M.append(times_M)
        print(f"  Added {n_epochs} EEG epochs from {file_name}")
        print(f"  Eye kept: L={len(kept_indicesL)}, R={len(kept_indicesR)}, M={len(kept_indicesM)}")

    except Exception as e:
        print(f"  Error processing {file_name}: {e}")

print("\nCombining all files...")

if len(all_X_eeg) == 0:
    raise RuntimeError("No valid files were processed.")

# Combine EEG
X_eeg = np.concatenate(all_X_eeg, axis=0)
y_cls = np.concatenate(all_y_cls, axis=0)
y_reg = np.concatenate(all_y_reg, axis=0)
subject_id_eeg = np.concatenate(all_subject_id_eeg, axis=0)
session_id_eeg = np.concatenate(all_session_id_eeg, axis=0)
event_times =np.concatenate(all_eeg_times)

# Combine Eye
X_eye_L = np.concatenate(all_X_eye_L, axis=0)
X_eye_R = np.concatenate(all_X_eye_R, axis=0)
X_eye_M = np.concatenate(all_X_eye_M, axis=0)

y_cls_eye_L = np.concatenate(all_y_cls_eye_L, axis=0)
y_cls_eye_R = np.concatenate(all_y_cls_eye_R, axis=0)
y_cls_eye_M = np.concatenate(all_y_cls_eye_M, axis=0)

y_reg_eye_L = np.concatenate(all_y_reg_eye_L, axis=0)
y_reg_eye_R = np.concatenate(all_y_reg_eye_R, axis=0)
y_reg_eye_M = np.concatenate(all_y_reg_eye_M, axis=0)
subject_id_eye_L =np.concatenate(all_subject_id_L,axis=0)
session_id_eye_L= np.concatenate(all_session_id_L, axis = 0)
subject_id_eye_R =np.concatenate(all_subject_id_R,axis=0)
session_id_eye_R= np.concatenate(all_session_id_R, axis = 0)
subject_id_eye_M =np.concatenate(all_subject_id_M,axis=0)
session_id_eye_M= np.concatenate(all_session_id_M, axis = 0)
times_L=np.concatenate(all_times_L, axis=0)
times_R=np.concatenate(all_times_R, axis=0)
times_M=np.concatenate(all_times_M, axis=0)
dataset_path_eeg = os.path.join(OUTPUT_DIR, "reve_eeg_dataset_epochs_before_after_event.npz")
dataset_path_eye = os.path.join(OUTPUT_DIR, "eye_dataset_befor_aftr_event.npz")

np.savez(
    dataset_path_eeg,
    X_eeg=X_eeg.astype(np.float32),
    y_cls=y_cls.astype(np.int64),
    y_reg=y_reg.astype(np.float32),
    subject_id_eeg=subject_id_eeg,
    session_id_eeg=session_id_eeg,
    ch_names=np.asarray(reference_ch_names, dtype=object),
    eeg_times =event_times,
)

np.savez(
    dataset_path_eye,
    X_eye_L=X_eye_L.astype(np.float32),
    X_eye_R=X_eye_R.astype(np.float32),
    X_eye_M=X_eye_M.astype(np.float32),
    y_cls_eye_L=y_cls_eye_L.astype(np.int64),
    y_cls_eye_R=y_cls_eye_R.astype(np.int64),
    y_cls_eye_M=y_cls_eye_M.astype(np.int64),
    y_reg_eye_L=y_reg_eye_L.astype(np.float32),
    y_reg_eye_R=y_reg_eye_R.astype(np.float32),
    y_reg_eye_M=y_reg_eye_M.astype(np.float32),
    subject_id_eye_L=subject_id_eye_L,
    subject_id_eye_R=subject_id_eye_R,
    subject_id_eye_M=subject_id_eye_M,
    session_id_eye_L=session_id_eye_L,
    session_id_eye_R=session_id_eye_R,
    session_id_eye_M=session_id_eye_M,
    times_L= times_L,
    times_R=times_R,
    times_M=times_M
)

print("Done.")
print(f"Saved EEG dataset to: {dataset_path_eeg}")
print(f"Saved eye dataset to: {dataset_path_eye}")

Found 30 files to process.

Processing: Copy of 08_25_2022_10_32_35-Exp_adadrive-Sbj_12-Ssn_01.dats-025.pkl


KeyboardInterrupt: 

In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import glob
import pickle
import gc
import numpy as np
import pandas as pd
import mne

from EEGPrep import finalPreprocesseeg
from EYEPrep import finalPreprocesseye


BASE_PATH = r"E:\koorathota_data\pkl_data\pkl_data\pkl_data"
OUTPUT_DIR = r"E:\koorathota_data\pkl_data\pkl_data\pkl_data\eegdata\indfile"

os.makedirs(OUTPUT_DIR, exist_ok=True)

pkl_files = glob.glob(os.path.join(BASE_PATH, "*.pkl"))

reference_ch_names = None
reference_shape = None

print(f"Found {len(pkl_files)} files to process.")


for file_path in pkl_files:
    file_name = os.path.basename(file_path)
    safe_name = os.path.splitext(file_name)[0]

    eeg_out_path = os.path.join(OUTPUT_DIR, f"{safe_name}_eeg.npz")
    eye_out_path = os.path.join(OUTPUT_DIR, f"{safe_name}_eye.npz")

    print(f"\nProcessing: {file_name}")

    try:
        with open(file_path, "rb") as f:
            data = pickle.load(f)

        resulteeg = finalPreprocesseeg(data)

        if resulteeg is None:
            raise ValueError(f"Missing required EEG keys in {file_name}")

        (
            raw_ica,
            epochs_ar,
            reject_log,
            filtered_left_peaks,
            filtered_right_peaks,
            filtered_left_onsets,
            filtered_right_onsets,
            events,
            y_class,
            y_reg,
            stats,
        ) = resulteeg

        resulteye = finalPreprocesseye(data, events)

        if resulteye is None:
            raise ValueError(f"Missing required eye keys in {file_name}")

        (
            pupil_epochsL,
            epoch_infoL,
            rejectedL,
            rel_timesL,
            kept_indicesL,
            pupil_epochsR,
            epoch_infoR,
            rejectedR,
            rel_timesR,
            kept_indicesR,
            pupil_epochsM,
            epoch_infoM,
            rejectedM,
            rel_timesM,
            kept_indicesM,
        ) = resulteye

        # Fill residual NaNs
        for name, arr in [
            ("left pupil", pupil_epochsL),
            ("right pupil", pupil_epochsR),
            ("mean pupil", pupil_epochsM),
        ]:
            nan_mask = ~np.isfinite(arr)
            if nan_mask.any():
                print(f"  Filling {nan_mask.sum()} residual NaN samples in {name} with 0")
                arr[nan_mask] = 0.0

        # Eye tensors: [N, T, 1]
        X_eye_L = pupil_epochsL[:, :, np.newaxis].astype(np.float32)
        X_eye_R = pupil_epochsR[:, :, np.newaxis].astype(np.float32)
        X_eye_M = pupil_epochsM[:, :, np.newaxis].astype(np.float32)

        n_epochs_L = X_eye_L.shape[0]
        n_epochs_R = X_eye_R.shape[0]
        n_epochs_M = X_eye_M.shape[0]

        # EEG labels
        keep_mask = ~reject_log.bad_epochs
        events_eeg = events[keep_mask]

        event_sample = events_eeg[:, 0].astype(np.int64)
        event_times = event_sample / epochs_ar.info["sfreq"]

        y_cls_eeg = np.asarray(y_class[keep_mask], dtype=np.int64)
        y_reg_eeg = np.asarray(y_reg[keep_mask], dtype=np.float32)

        # Eye labels
        y_cls_eye_L = np.asarray(y_class[kept_indicesL], dtype=np.int64)
        y_cls_eye_R = np.asarray(y_class[kept_indicesR], dtype=np.int64)
        y_cls_eye_M = np.asarray(y_class[kept_indicesM], dtype=np.int64)

        y_reg_eye_L = np.asarray(y_reg[kept_indicesL], dtype=np.float32)
        y_reg_eye_R = np.asarray(y_reg[kept_indicesR], dtype=np.float32)
        y_reg_eye_M = np.asarray(y_reg[kept_indicesM], dtype=np.float32)

        times_L = np.array([d["event_time"] for d in epoch_infoL], dtype=np.float64)
        times_R = np.array([d["event_time"] for d in epoch_infoR], dtype=np.float64)
        times_M = np.array([d["event_time"] for d in epoch_infoM], dtype=np.float64)

        # EEG tensor
        epochs_reve = epochs_ar.copy().pick("eeg")
        epochs_reve = epochs_reve.resample(200)

        X_eeg = epochs_reve.get_data(copy=True).astype(np.float32)
        ch_names = epochs_reve.ch_names
        n_epochs = X_eeg.shape[0]

        if len(y_cls_eeg) != n_epochs or len(y_reg_eeg) != n_epochs:
            raise ValueError(
                f"Label length mismatch in {file_name}: "
                f"n_epochs={n_epochs}, "
                f"len(y_cls_eeg)={len(y_cls_eeg)}, "
                f"len(y_reg_eeg)={len(y_reg_eeg)}"
            )

        if reference_ch_names is None:
            reference_ch_names = ch_names
        elif ch_names != reference_ch_names:
            raise ValueError(f"Channel mismatch in {file_name}")

        if reference_shape is None:
            reference_shape = X_eeg.shape[1:]
        elif X_eeg.shape[1:] != reference_shape:
            raise ValueError(
                f"Shape mismatch in {file_name}: "
                f"{X_eeg.shape[1:]} vs {reference_shape}"
            )

        # Subject/session metadata
        info = data["Unity_TrialInfo"]
        info_t = info[0]
        idd = info_t[0]
        session = info_t[1]

        subj_value = idd[0]
        sess_value = session[0]

        subject_id_eeg = np.full(n_epochs, subj_value, dtype=np.int64)
        session_id_eeg = np.full(n_epochs, sess_value, dtype=np.int64)

        subject_id_eye_L = np.full(n_epochs_L, subj_value, dtype=np.int64)
        session_id_eye_L = np.full(n_epochs_L, sess_value, dtype=np.int64)

        subject_id_eye_R = np.full(n_epochs_R, subj_value, dtype=np.int64)
        session_id_eye_R = np.full(n_epochs_R, sess_value, dtype=np.int64)

        subject_id_eye_M = np.full(n_epochs_M, subj_value, dtype=np.int64)
        session_id_eye_M = np.full(n_epochs_M, sess_value, dtype=np.int64)

        # Save EEG for this file
        np.savez_compressed(
            eeg_out_path,
            X_eeg=X_eeg,
            y_cls=y_cls_eeg,
            y_reg=y_reg_eeg,
            subject_id_eeg=subject_id_eeg,
            session_id_eeg=session_id_eeg,
            ch_names=np.asarray(ch_names, dtype=object),
            eeg_times=event_times.astype(np.float64),
        )

        # Save eye data for this file
        np.savez_compressed(
            eye_out_path,
            X_eye_L=X_eye_L,
            X_eye_R=X_eye_R,
            X_eye_M=X_eye_M,

            y_cls_eye_L=y_cls_eye_L,
            y_cls_eye_R=y_cls_eye_R,
            y_cls_eye_M=y_cls_eye_M,

            y_reg_eye_L=y_reg_eye_L,
            y_reg_eye_R=y_reg_eye_R,
            y_reg_eye_M=y_reg_eye_M,

            subject_id_eye_L=subject_id_eye_L,
            subject_id_eye_R=subject_id_eye_R,
            subject_id_eye_M=subject_id_eye_M,

            session_id_eye_L=session_id_eye_L,
            session_id_eye_R=session_id_eye_R,
            session_id_eye_M=session_id_eye_M,

            times_L=times_L,
            times_R=times_R,
            times_M=times_M,
        )

        print(f"  Saved EEG: {eeg_out_path}")
        print(f"  Saved eye: {eye_out_path}")
        print(f"  EEG epochs: {n_epochs}")
        print(f"  Eye kept: L={len(kept_indicesL)}, R={len(kept_indicesR)}, M={len(kept_indicesM)}")

    except Exception as e:
        print(f"  Error processing {file_name}: {e}")

    finally:
        # Free memory after each file
        vars_to_delete = [
            "data", "resulteeg", "resulteye",
            "raw_ica", "epochs_ar", "reject_log", "epochs_reve",
            "X_eeg", "X_eye_L", "X_eye_R", "X_eye_M",
            "pupil_epochsL", "pupil_epochsR", "pupil_epochsM",
            "y_cls_eeg", "y_reg_eeg",
            "y_cls_eye_L", "y_cls_eye_R", "y_cls_eye_M",
            "y_reg_eye_L", "y_reg_eye_R", "y_reg_eye_M",
        ]

        for var in vars_to_delete:
            if var in locals():
                del locals()[var]

        gc.collect()


print("\nDone.")
print(f"Saved per-file datasets in: {OUTPUT_DIR}")

C:\Users\USER\anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


Found 30 files to process.

Processing: Copy of 08_25_2022_10_32_35-Exp_adadrive-Sbj_12-Ssn_01.dats-025.pkl
Creating RawArray with float64 data, n_channels=89, n_times=3122151
    Range : 0 ... 3122150 =      0.000 ...  1524.487 secs
Ready.
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 55 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 55.00 Hz
- Upper transition bandwidth: 13.75 Hz (-6 dB cutoff frequency: 61.88 Hz)
- Filter length: 6759 samples (3.300 s)



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.3s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.7s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    1.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    1.3s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  64 out of  64 | elapsed:   17.8s finished


Trigger channel TRIGGER has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)


C:\Users\USER\Documents\diplwmatikh\testakia\maiin\EEGPrep.py:57: RuntimeWarning: Trigger channel contains negative values, using absolute value. If data were acquired on a Neuromag system with STI016 active, consider using uint_cast=True to work around an acquisition bug
  raw.resample(128.0, npad = "auto")


...

C:\Users\USER\Documents\diplwmatikh\testakia\maiin\EEGPrep.py:57: RuntimeWarning: Trigger channel contains negative values, using absolute value. If data were acquired on a Neuromag system with STI016 active, consider using uint_cast=True to work around an acquisition bug
  raw.resample(128.0, npad = "auto")


Trigger channel TRIGGER has a non-zero initial value of {initial_value} (consider using initial_event=True to detect this event)


C:\Users\USER\Documents\diplwmatikh\testakia\maiin\EEGPrep.py:57: RuntimeWarning: Trigger channel contains negative values, using absolute value. If data were acquired on a Neuromag system with STI016 active, consider using uint_cast=True to work around an acquisition bug
  raw.resample(128.0, npad = "auto")


  Error processing Copy of 08_26_2022_11_53_52-Exp_adadrive-Sbj_12-Ssn_02.dats-028.pkl: Unable to allocate 2.58 GiB for an array with shape (89, 3891602) and data type float64

Processing: Copy of 09_02_2022_10_34_48-Exp_adadrive-Sbj_12-Ssn_03.dats.pkl


KeyboardInterrupt: 

In [3]:
import psutil
mem = psutil.virtual_memory()

print("Total GB:", round(mem.total / 1e9, 2))
print("Available GB:", round(mem.available / 1e9, 2))
print("Used %:", mem.percent)

Total GB: 12.8
Available GB: 2.3
Used %: 82.1


In [4]:
import sys, platform, os

print("Python:", sys.executable)
print("Platform:", platform.platform())
print("Processor:", platform.processor())
print("Username:", os.getlogin())

Python: C:\Users\USER\anaconda3\python.exe
Platform: Windows-10-10.0.19045-SP0
Processor: Intel64 Family 6 Model 142 Stepping 10, GenuineIntel
Username: USER


In [5]:
!hostname

DESKTOP-KEEM6TB
